In [1]:
import json

with open("school_codes_list.json", "r") as f:
    school_codes_list = json.load(f)

In [2]:
import pandas as pd
import requests
from io import StringIO

url = "https://www3.cde.ca.gov/demo-downloads/cgr/cgr12mo23.txt"

r = requests.get(url)
r.raise_for_status()
text = r.content.decode("cp1252", errors="replace")

In [3]:
df = pd.read_csv(
    StringIO(text),
    sep="\t",
    dtype={
        "AcademicYear": "string",
        "AggregateLevel": "string",
        "CountyCode": "string",
        "DistrictCode": "string",
        "SchoolCode": "string",
        "CountyName": "string",
        "DistrictName": "string",
        "SchoolName": "string",
        "CharterSchool": "string",
        "AlternativeSchoolAccountabilityStatus": "string",
        "ReportingCategory": "string",
        "CompleterType": "string"
    },
    keep_default_na=False
)

In [9]:
df.columns = df.columns.str.strip()

string_cols = [
    "AcademicYear", "AggregateLevel", "CountyCode", "DistrictCode", "SchoolCode",
    "CountyName", "DistrictName", "SchoolName", "CharterSchool",
    "AlternativeSchoolAccountabilityStatus", "ReportingCategory", "CompleterType"
]

for col in string_cols:
    df[col] = df[col].str.strip()

# keep only school-level total rows
college_df = df[df["AggregateLevel"] == "S"].copy()
college_df = college_df[college_df["ReportingCategory"] == "TA"].copy()
college_df = college_df[college_df["CompleterType"] == "TA"].copy()

# format codes
college_df["SchoolCode"] = college_df["SchoolCode"].astype(int).astype(str).str.zfill(7)

# filter to your schools
college_df = college_df[college_df["SchoolCode"].isin(school_codes_list)].copy()
college_df = college_df.reset_index(drop=True)

In [10]:
college_df = college_df[['SchoolCode', 'College Going Rate - Total (12 Months)']]

In [11]:
college_df

,SchoolCode,College Going Rate - Total (12 Months)
0,0130625,26.8
1,0130229,82.8
2,0106401,80.5
3,0130450,81.5
4,0131177,79.4
...,...,...
1368,0101162,69.8
1369,5830047,*
1370,5835202,60.4
1371,5830013,60.1


In [ ]:
college_df.to_csv('../cleaned_data/CGR_data.csv')